# Fresh Start NBA — Notebook 4: XGBoost Interactive Training

Retrain your XGBoost models interactively. Tune hyperparameters, compare results against
your current production models, and export new `.pkl` files back to Drive.

**Prerequisites:** Run notebook 3 first to generate `outputs/nba_features.csv`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE_DIR   = Path('/content/drive/MyDrive/Fresh_Start_NBA_Colab')
DATA_DIR   = BASE_DIR / 'data'
MODELS_DIR = BASE_DIR / 'models'
OUT_DIR    = BASE_DIR / 'outputs'

sns.set_theme(style='darkgrid')
plt.rcParams['figure.dpi'] = 110
print(f'XGBoost version: {xgb.__version__}')

## Load Feature-Engineered Data

In [ ]:
feat_path = OUT_DIR / 'nba_features.csv'
if not feat_path.exists():
    raise FileNotFoundError(
        'nba_features.csv not found. Run notebook 3 first to generate it.'
    )

df = pd.read_csv(feat_path, low_memory=False)
df['game_date'] = pd.to_datetime(df['game_date'])
df = df.sort_values('game_date').reset_index(drop=True)
print(f'Loaded features: {df.shape}')

# Load feature column list
with open(BASE_DIR / 'feature_cols_advanced.json') as f:
    feat_meta = json.load(f)
ALL_FEATURE_COLS = feat_meta['feature_columns']
FEATURE_COLS = [c for c in ALL_FEATURE_COLS if c in df.columns]
print(f'Feature cols available: {len(FEATURE_COLS)}/{len(ALL_FEATURE_COLS)}')

## Hyperparameter Configuration

Edit these to experiment with different model settings per stat.

In [ ]:
# ── Tune these values and re-run the training cell below ─────────────────────
# These mirror the production values in train_advanced_models.py.
# Try adjusting max_depth, n_estimators, or learning_rate for a stat and see
# what happens to MAE and OVER/UNDER accuracy.

REGRESSION_PARAMS = {
    'pts': dict(max_depth=7, learning_rate=0.04, n_estimators=400,
                subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
                reg_alpha=0.05, reg_lambda=1.0),
    'trb': dict(max_depth=6, learning_rate=0.05, n_estimators=300,
                subsample=0.8, colsample_bytree=0.7, min_child_weight=3,
                reg_alpha=0.10, reg_lambda=1.0),
    'ast': dict(max_depth=5, learning_rate=0.05, n_estimators=300,
                subsample=0.7, colsample_bytree=0.7, min_child_weight=5,
                reg_alpha=0.10, reg_lambda=1.5),
    'pa':  dict(max_depth=6, learning_rate=0.05, n_estimators=350,
                subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
                reg_alpha=0.10, reg_lambda=1.0),
    'tov': dict(max_depth=5, learning_rate=0.05, n_estimators=250,
                subsample=0.7, colsample_bytree=0.7, min_child_weight=5,
                reg_alpha=0.20, reg_lambda=2.0),
}

# Season sample weights — recent games matter more
SEASON_WEIGHTS = {
    '2025-26': (pd.Timestamp('2025-10-01'), None,                       3.0),
    '2024-25': (pd.Timestamp('2024-10-01'), pd.Timestamp('2025-10-01'), 2.0),
    '2023-24': (None,                       pd.Timestamp('2024-10-01'), 1.0),
}

# Which stats to train — remove any you don't need
STATS_TO_TRAIN = ['pts', 'trb', 'ast']

# 30-day holdout for evaluation (matches production logic)
HOLDOUT_DAYS = 30

print('Config ready. Run the training cell below.')

## Helper Functions

In [ ]:
def get_sample_weights(df_subset):
    dates = pd.to_datetime(df_subset['game_date'])
    w = np.ones(len(df_subset))
    for season, (lo, hi, weight) in SEASON_WEIGHTS.items():
        mask = np.ones(len(df_subset), dtype=bool)
        if lo is not None:
            mask &= (dates >= lo).values
        if hi is not None:
            mask &= (dates < hi).values
        w[mask] = weight
    return w


def train_stat_model(df, stat, params, feature_cols):
    """Train regression + OVER/UNDER classifier for one stat. Returns metrics."""
    # Require at least 10 past games per player
    MIN_GAMES = 10
    df_clean = df.dropna(subset=[stat] + [c for c in feature_cols if c in df.columns]).copy()
    df_clean = df_clean[df_clean['games_played'] >= MIN_GAMES]

    if len(df_clean) < 500:
        print(f'  [{stat}] Not enough rows ({len(df_clean)}) — skipping')
        return None, None, {}

    # Train / holdout split
    cutoff = df_clean['game_date'].max() - pd.Timedelta(days=HOLDOUT_DAYS)
    train_df = df_clean[df_clean['game_date'] <= cutoff]
    test_df  = df_clean[df_clean['game_date'] >  cutoff]

    avail_feats = [c for c in feature_cols if c in df_clean.columns]
    X_train = train_df[avail_feats].fillna(0)
    y_train = train_df[stat].values
    X_test  = test_df[avail_feats].fillna(0)
    y_test  = test_df[stat].values

    weights = get_sample_weights(train_df)

    # Regression model
    reg = xgb.XGBRegressor(
        objective='reg:squarederror',
        tree_method='hist',
        device='cuda' if xgb.__version__ >= '2.0' else 'cpu',
        random_state=42,
        **params
    )
    reg.fit(X_train, y_train, sample_weight=weights,
            eval_set=[(X_test, y_test)], verbose=False)

    preds = reg.predict(X_test)
    mae = mean_absolute_error(y_test, preds)

    # OVER/UNDER classifier (use L10 rolling avg as proxy line when no real line)
    line_col = f'{stat}_l10'
    if line_col in test_df.columns:
        real_line_col = f'{stat}_real_line'
        if real_line_col in test_df.columns:
            lines_test = test_df[real_line_col].fillna(test_df[line_col]).values
        else:
            lines_test = test_df[line_col].values
        ou_labels = (y_test > lines_test).astype(int)
        ou_preds  = (preds > lines_test).astype(int)
        accuracy = (ou_labels == ou_preds).mean()
    else:
        accuracy = None

    metrics = {
        'n_train': len(train_df),
        'n_test': len(test_df),
        'mae': float(mae),
        'accuracy': float(accuracy) if accuracy is not None else None,
        'n_features': len(avail_feats)
    }
    return reg, avail_feats, metrics


print('Helper functions ready.')

## Train Models

This cell trains one regression model per stat in `STATS_TO_TRAIN`.
Wait for the output — it will print MAE and accuracy for each stat when done.

In [ ]:
trained_models = {}
new_results = {}

for stat in STATS_TO_TRAIN:
    params = REGRESSION_PARAMS.get(stat, REGRESSION_PARAMS['pts'])
    print(f'Training {stat.upper()}...')
    model, feat_cols_used, metrics = train_stat_model(df, stat, params, FEATURE_COLS)

    if model is not None:
        trained_models[stat] = {'model': model, 'features': feat_cols_used}
        new_results[stat] = metrics
        print(f'  MAE: {metrics["mae"]:.3f}  |  '
              f'Accuracy: {metrics["accuracy"]*100:.1f}%  |  '
              f'Train rows: {metrics["n_train"]:,}  Test rows: {metrics["n_test"]:,}')

print('\nDone.')

## Compare vs Production Models

In [ ]:
# Load production results from Drive
prod_results_path = MODELS_DIR / 'results.json'

if prod_results_path.exists():
    with open(prod_results_path) as f:
        prod_results = json.load(f)
else:
    prod_results = {}
    print('Production results.json not found — upload models/ to Drive.')

comparison_rows = []
for stat in STATS_TO_TRAIN:
    if stat not in new_results:
        continue
    prod = prod_results.get(stat, {})
    new  = new_results[stat]
    comparison_rows.append({
        'stat': stat,
        'prod_mae':  prod.get('mae'),
        'new_mae':   new['mae'],
        'mae_delta': new['mae'] - prod.get('mae', new['mae']),
        'prod_acc':  prod.get('accuracy'),
        'new_acc':   new['accuracy'],
        'acc_delta': (new['accuracy'] or 0) - (prod.get('accuracy') or 0)
    })

comp_df = pd.DataFrame(comparison_rows)
print('\n=== New vs Production Models ===')
print(comp_df.to_string(index=False, float_format=lambda x: f'{x:.4f}' if x else 'N/A'))

# Visual comparison
if len(comp_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    x = np.arange(len(comp_df))
    w = 0.35

    axes[0].bar(x - w/2, comp_df['prod_mae'], w, label='Production', color='steelblue', alpha=0.8)
    axes[0].bar(x + w/2, comp_df['new_mae'],  w, label='New (Colab)', color='coral', alpha=0.8)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(comp_df['stat'].str.upper())
    axes[0].set_title('MAE: Production vs New', fontweight='bold')
    axes[0].set_ylabel('MAE (lower = better)')
    axes[0].legend()

    axes[1].bar(x - w/2, (comp_df['prod_acc'].fillna(0)) * 100, w, label='Production', color='steelblue', alpha=0.8)
    axes[1].bar(x + w/2, (comp_df['new_acc'].fillna(0)) * 100,  w, label='New (Colab)', color='coral', alpha=0.8)
    axes[1].axhline(50, color='black', linestyle='--', linewidth=1)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(comp_df['stat'].str.upper())
    axes[1].set_title('OVER/UNDER Accuracy: Production vs New', fontweight='bold')
    axes[1].set_ylabel('Accuracy %')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(OUT_DIR / 'model_comparison.png', bbox_inches='tight')
    plt.show()

## Feature Importance for New Models

In [ ]:
for stat, model_info in trained_models.items():
    model = model_info['model']
    feats = model_info['features']

    scores = model.feature_importances_
    fi_df = pd.DataFrame({'feature': feats, 'importance': scores})
    fi_df = fi_df.sort_values('importance', ascending=False).head(15)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(fi_df['feature'][::-1], fi_df['importance'][::-1],
            color=sns.color_palette('viridis', len(fi_df)))
    ax.set_title(f'{stat.upper()} — Top 15 Feature Importances (new model)', fontweight='bold')
    ax.set_xlabel('Importance (gain)')
    plt.tight_layout()
    plt.savefig(OUT_DIR / f'fi_new_{stat}.png', bbox_inches='tight')
    plt.show()

## Export New Models to Drive

Only run this if the new models beat production. Download the `.pkl` files from Drive
and replace the ones in your local `models/` folder to deploy.

In [ ]:
# Confirm you want to export before running
EXPORT_NEW_MODELS = False   # set to True to save

if EXPORT_NEW_MODELS:
    export_dir = BASE_DIR / 'models_new'
    export_dir.mkdir(exist_ok=True)

    for stat, model_info in trained_models.items():
        out_path = export_dir / f'xgb_{stat}_advanced.pkl'
        with open(out_path, 'wb') as f:
            pickle.dump(model_info['model'], f)
        print(f'Saved: {out_path}')

    # Save updated results.json
    with open(export_dir / 'results.json', 'w') as f:
        json.dump({s: {'mae': r['mae'], 'accuracy': r['accuracy']}
                   for s, r in new_results.items()}, f, indent=2)
    print(f'Saved updated results.json to {export_dir}')
    print('\nDownload these from Drive and replace your local models/ folder to deploy.')
else:
    print('EXPORT_NEW_MODELS is False — set it to True in this cell to save.')